In [ ]:
import pandas as pd
from google import genai # <-- Menggunakan library baru
import json
import os
from dotenv import load_dotenv

load_dotenv()

api=os.getenv("API")

# Inisialisasi client Gemini versi terbaru
# Ganti dengan API Key yang kamu dapatkan dari Google AI Studio
client = genai.Client(api_key=api)

def get_full_llm_prediction(text):
    # ==========================================
    # PROMPT ENGINEERING: BEBAN KERJA 100% KE LLM
    # ==========================================
    prompt = f"""
    Kamu adalah asisten AI ahli Aspect-Based Sentiment Analysis (ABSA).
    Tugasmu adalah membaca teks ulasan pengguna, lalu mengekstrak Aspek, Opini, dan langsung menentukan Sentimennya.
    
    Aturan ketat:
    1. Ekstrak SEGALA aspek yang memiliki opini (makanan, minuman, pelayanan, suasana, kebersihan, dll).
    2. Pisahkan aspek yang dirangkai dengan kata hubung.
    3. Tangkap opini secara utuh (termasuk kata penegas/negasi seperti "too sweet", "not good").
    4. Tentukan sentimen dari opini tersebut. Pilihan Sentimen HANYA: "Positif" atau "Negatif".
    5. Kembalikan HANYA dalam format JSON array. Tidak boleh ada teks pengantar atau penutup.
    
    Contoh Output:
    [
        {{"aspek": "cake", "opini": "too sweet", "sentimen": "Negatif"}},
        {{"aspek": "waiter", "opini": "very friendly", "sentimen": "Positif"}}
    ]

    Teks Ulasan: "{text}"
    """
    
    try:
        # Memanggil Gemini 2.5 Flash
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt
        )
        
        # Membersihkan output
        clean_json = response.text.replace('```json', '').replace('```', '').strip()
        data_ekstraksi = json.loads(clean_json)
        
        hasil_akhir = []
        
        # Merakit hasil langsung dari JSON LLM (Tanpa DistilBERT)
        for item in data_ekstraksi:
            aspect = item.get("aspek", "")
            opinion = item.get("opini", "")
            sentiment = item.get("sentimen", "")
            
            if aspect and opinion:
                hasil_akhir.append({
                    "Aspek (Target)": str(aspect).capitalize(),
                    "Opini Pengguna": str(opinion).lower(),
                    "Sentimen (LLM)": sentiment, 
                    "Keyakinan Model": "N/A (Generative AI)" # LLM tidak mengeluarkan skor probabilitas klasifikasi
                })
                
        if not hasil_akhir:
             return pd.DataFrame([{"Pesan": "Gagal mengekstrak informasi dari ulasan."}])
             
        return pd.DataFrame(hasil_akhir)
        
    except Exception as e:
        return pd.DataFrame([{"Pesan": f"Terjadi kesalahan LLM: {str(e)}"}])

In [5]:
# ==========================================
# UJI COBA OPSI FULL LLM
# ==========================================
ulasan_user = "The AC is very cold and the waiters are friendly, but the steak is tough."
df_hasil_full = get_full_llm_prediction(ulasan_user)
display(df_hasil_full)

,Aspek (Target),Opini Pengguna,Sentimen (LLM),Keyakinan Model
0,Ac,very cold,Negatif,N/A (Generative AI)
1,Waiters,friendly,Positif,N/A (Generative AI)
2,Steak,tough,Negatif,N/A (Generative AI)
